# 01 Get the data ready

Nothing clever happens here. This notebook downloads the lab data the two models learn from, turns every molecule into numbers, and saves the result so the other notebooks can start straight from it.

**What we are downloading, and why each one:**

| File | What a row says | Used for |
| --- | --- | --- |
| `CYP3A4_Veith` | this molecule was lab-tested, and it does / does not jam CYP3A4 | training Model A |
| `CYP3A4_Substrate_CarbonMangels` | this drug is / is not cleared by CYP3A4 | condition 2 of the screen |
| `hERG` | this molecule does / does not block the hERG channel | training Model B |

**The words, once:**

- **CYP3A4** is a liver enzyme. Its job is to break drugs down so the body can get rid of them. It handles roughly half of all prescription drugs.
- **Jamming it** (the technical word is *inhibition*) means a drug that relies on it stops being cleared, so it builds up in the blood. A normal dose quietly becomes too much.
- **hERG** is a channel in heart cells that lets the heart reset electrically between beats. Block it and the reset drags, which shows up on an ECG as a long **QT interval** and can turn into a dangerous rhythm.

We also check how many of each dataset's answers are yes vs no. That matters because the hERG set is small and lopsided, and that is what forces Model B to be checked differently from Model A.

In [1]:
import sys
import importlib
import os
import urllib.request

# Let the notebook import the helpers that live in src/.
sys.path.append(os.path.abspath(os.path.join("..", "src")))

import numpy as np
import pandas as pd
import features

RAW_DIR = os.path.join("..", "data", "raw")
PROCESSED_DIR = os.path.join("..", "data", "processed")
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("fingerprint setting: radius", features.FP_RADIUS, "size", features.FP_SIZE)
print("features per molecule:", features.N_FEATURES,
      "=", features.FP_SIZE, "fragment counts +", features.N_DESCRIPTORS, "descriptors")

# Python only reads a file from src/ the FIRST time it is imported. After that
# it serves a cached copy, so any edit you make to src/ is invisible to a
# notebook that has already run this cell.
#
# The symptom is confusing: an AttributeError saying a function does not exist,
# when you can see it sitting right there in the file. Reloading here means
# "Run All" always uses what is actually on disk.
for module in (features,):
    importlib.reload(module)

fingerprint setting: radius 2 size 2048
features per molecule: 2058 = 2048 fragment counts + 10 descriptors


## Download the three files

These datasets come from TDC (Therapeutics Data Commons, a public collection of drug
datasets). They are published as plain text files on Harvard Dataverse, so we just
download them directly.

**Why not use the `tdc` python package instead?** It demands older versions of numpy,
pandas and RDKit than this project uses, and on Windows installing it pulls in a package
that has no ready-made Windows build and fails to compile. Three plain downloads avoid
the entire mess.

`data/` is not tracked by git, so this downloads once on a fresh copy of the project and
skips every time after that.

In [2]:
# The file ID numbers come from the TDC package's own list, so these point at
# exactly the datasets the package would have fetched.
TDC_FILES = {
    "cyp3a4_veith.tab": "https://dataverse.harvard.edu/api/access/datafile/4259582",
    "herg.tab": "https://dataverse.harvard.edu/api/access/datafile/4259588",
    # This one is condition 2 of the screen: 670 drugs that were lab-tested for
    # whether CYP3A4 clears them, with structures included. Having it means the
    # drug side of the project needs no sign-up-required download and no
    # structures typed in by hand.
    "cyp3a4_substrate_carbonmangels.tab":
        "https://dataverse.harvard.edu/api/access/datafile/4259581",
}


def download(url, destination):
    # The server rejects Python's default request with "403 Forbidden", so we
    # tell it we are an ordinary browser instead. Nothing sneaky, the files are
    # public; the server just refuses anything that does not look like a browser.
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=120) as response:
        data = response.read()
    # Only write the file once the whole download finished. Otherwise a dropped
    # connection leaves a half-written file that looks complete next time, and
    # the error turns up much later as confusing nonsense in the data.
    with open(destination, "wb") as handle:
        handle.write(data)
    return len(data)


for filename, url in TDC_FILES.items():
    destination = os.path.join(RAW_DIR, filename)
    if os.path.exists(destination):
        print("already have", filename)
        continue
    print("downloading", filename, "...")
    print("   saved", download(url, destination), "bytes to", destination)

already have cyp3a4_veith.tab
already have herg.tab
already have cyp3a4_substrate_carbonmangels.tab


## Model A's data: which molecules jam the enzyme

Each row is one molecule that was actually tested in a lab, with a yes/no answer.

In [3]:
# About 12,000 molecules, all tested in the same lab screen.
cyp_df = pd.read_csv(os.path.join(RAW_DIR, "cyp3a4_veith.tab"), sep="\t")

print("CYP3A4_Veith shape:", cyp_df.shape)
print("columns:", list(cyp_df.columns))
cyp_df.head()

CYP3A4_Veith shape: (12328, 3)
columns: ['Drug_ID', 'Drug', 'Y']


,Drug_ID,Drug,Y
0,6602638.0,CCCC(=O)Nc1ccc(N2CCN(CC)CC2)c(Cl)c1.Cl,0
1,644510.0,O=c1[nH]c2cc3c(cc2cc1CN(CCCO)Cc1nnnn1Cc1ccc(F)...,1
2,644675.0,CC(=O)N(c1ccc2oc(=O)sc2c1)S(=O)(=O)c1cccs1,1
3,644890.0,COc1ccccc1C(c1nnnn1C(C)(C)C)N1CCN(Cc2ccncc2)CC1,1
4,645063.0,CC(=O)Nc1cccc(NC(=O)C2CCCN2C(=O)Nc2ccccc2C)c1,0


In [4]:
# How many yes vs no? This decides whether "accuracy" would be a lying number.
# If 95% of answers were "no", a model that always says no scores 95% and has
# learned nothing.
print("CYP3A4 label counts:")
print(cyp_df["Y"].value_counts())
print("positive fraction:", round(float(cyp_df["Y"].mean()), 3))

CYP3A4 label counts:
Y
0    7218
1    5110
Name: count, dtype: int64
positive fraction: 0.415


## Model B's data: which molecules block the heart channel

Far fewer molecules here, and that shortage shapes how Model B has to be checked.

In [5]:
# Only a few hundred molecules. That is why Model B gets cross validation
# (test on five different slices and average) instead of one single split:
# with a test set this small, one split's score depends too much on luck.
herg_df = pd.read_csv(os.path.join(RAW_DIR, "herg.tab"), sep="\t")

print("hERG shape:", herg_df.shape)
print("columns:", list(herg_df.columns))
print("hERG label counts:")
print(herg_df["Y"].value_counts())
print("positive fraction:", round(float(herg_df["Y"].mean()), 3))

hERG shape: (655, 3)
columns: ['Drug_ID', 'Drug', 'Y']
hERG label counts:
Y
1.0    451
0.0    204
Name: count, dtype: int64
positive fraction: 0.689


## Turn the molecules into numbers

A model cannot read a molecule. Each one has to become a fixed list of numbers first, which is what `src/features.py` does.

A few molecules in any public dataset are written incorrectly and cannot be read at all. Those get dropped, so the function also returns **which rows survived**. We have to use that list to drop the matching answers too, otherwise molecule 7's structure ends up paired with molecule 5's answer and everything downstream is quietly wrong.

In [6]:
X_cyp, kept_cyp = features.smiles_list_to_matrix(cyp_df["Drug"].tolist())
y_cyp = cyp_df["Y"].values[kept_cyp]

print("CYP features:", X_cyp.shape)
print("dropped as unparseable:", len(cyp_df) - len(kept_cyp))

# Count how many different fragments show up. Careful: only the first 2048
# columns are fragments. The last ten are things like molecular weight, so
# adding up the whole row would mix "number of rings" with "weighs 340" and
# produce a number that means nothing.
print("mean distinct fragments per molecule:",
      round(float(features.bits_from_matrix(X_cyp).sum(axis=1).mean()), 1))

CYP features: (12328, 2058)
dropped as unparseable: 0
mean distinct fragments per molecule: 43.7


In [7]:
X_herg, kept_herg = features.smiles_list_to_matrix(herg_df["Drug"].tolist())
y_herg = herg_df["Y"].values[kept_herg]

print("hERG features:", X_herg.shape)
print("dropped as unparseable:", len(herg_df) - len(kept_herg))
print("mean distinct fragments per molecule:",
      round(float(features.bits_from_matrix(X_herg).sum(axis=1).mean()), 1))

hERG features: (655, 2058)
dropped as unparseable: 0
mean distinct fragments per molecule: 46.0


In [8]:
# Save, so notebooks 02 and 03 can start from here instead of redoing all the
# featurising every time.
np.save(os.path.join(PROCESSED_DIR, "X_cyp3a4.npy"), X_cyp)
np.save(os.path.join(PROCESSED_DIR, "y_cyp3a4.npy"), y_cyp)
np.save(os.path.join(PROCESSED_DIR, "X_herg.npy"), X_herg)
np.save(os.path.join(PROCESSED_DIR, "y_herg.npy"), y_herg)

# Keep the original molecule text too. Notebook 04 has to go backwards from
# "fragment 1536 mattered" to a picture of the actual atoms, and that is
# impossible from the numbers alone.
cyp_df.iloc[kept_cyp].to_csv(os.path.join(PROCESSED_DIR, "cyp3a4_clean.csv"), index=False)
herg_df.iloc[kept_herg].to_csv(os.path.join(PROCESSED_DIR, "herg_clean.csv"), index=False)
print("saved to", PROCESSED_DIR)

saved to ..\data\processed
